# 01 · Multimodal chat — read a photo, not just a question

Contoso Outdoors customer support gets emails with photos attached all the time — a customer isn't sure which tent they own, or wants to know if a product suits their trip, and they just snap a picture instead of typing the model name. A **multimodal** model can look at that photo *and* read the question in the same request.

In this notebook you'll send a real product photo to two different models and compare how each one reads it — then use vision to produce a structured triage record a support system could act on automatically.

### Learning objectives
By the end of this notebook you'll be able to:
- **Send an image + text prompt** to a vision-capable model and read back a grounded-sounding answer.
- **Compare** how `gpt-5.4` and `claude-sonnet-4-6` each interpret the *same* photo and question.
- **Extract structured data** (a JSON triage record) from an image — the pattern behind automated support routing.
- **Recognize the limit**: this is chat, not the grounded TrailMate agent — the model answers from training knowledge, not from the real manuals.

> ⏱️ **~20 minutes** · Optional deep dive — use this notebook as a sandbox to try your own images and prompts.


## 1 · Set up and a helper to ask about an image

Same `.env` you validated in the core labs — no new setup. We add one small helper, `ask_about_image(...)`, that sends a **photo + a text prompt** together and returns the answer, plus which model actually answered.


In [ ]:
import base64
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient


def find_agent_builder() -> Path:
    """Locate foundry/agent-builder from anywhere in the tree (repo root or labs/more)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "foundry" / "agent-builder" / "src").is_dir():
            return base / "foundry" / "agent-builder"
        if base.name == "agent-builder" and (base / "src").is_dir():
            return base
    raise FileNotFoundError("Run labs/core/00-validate-setup.ipynb first — src/.env not found.")


AB = find_agent_builder()
load_dotenv(AB / "src" / ".env")
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()


def ask_about_image(model: str, image_path: Path, prompt: str) -> dict:
    """Send an image + text prompt to a vision-capable model, return the answer."""
    # Images go in as a base64 data URL alongside the text — one message, two inputs.
    image_b64 = base64.b64encode(image_path.read_bytes()).decode()
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            ],
        }],
    )
    return {
        "asked_model": model,
        "served_model": resp.model,
        "answer": resp.choices[0].message.content,
    }


print("Ready. `ask_about_image(model, image_path, prompt)` sends a photo + question and returns the answer.")


## 2 · The scenario — a customer sends a photo, not a model name

A customer emails support a photo of their tent with the question: *"Is this waterproof enough for a rainy weekend trip?"* They don't know the model name — just the photo.

> ❓ **Can `gpt-5.4` identify the product from the photo and give a useful answer, without being told what it is?**

Let's find out. We use a real Contoso Outdoors product photo as the stand-in for the customer's picture.


In [ ]:
from IPython.display import Image, display

CUSTOMER_PHOTO = AB / "assets" / "products" / "alpine-explorer-tent.png"
CUSTOMER_QUESTION = (
    "A customer emailed us this photo and asked: 'Is this tent waterproof enough "
    "for a rainy weekend trip?' Identify the product if you can, and answer their "
    "question using what you know about it."
)

print("The customer's photo:")
display(Image(filename=str(CUSTOMER_PHOTO)))

gpt_result = ask_about_image("gpt-5.4", CUSTOMER_PHOTO, CUSTOMER_QUESTION)
print(f"\n--- {gpt_result['asked_model']} answered ---")
print(gpt_result["answer"])


## 3 · Compare with Claude Sonnet 4.6

Same photo, same question — this time to `claude-sonnet-4-6`. Multimodal isn't a GPT-only trick; Claude reads images too.

> ❓ **Your call:** Read both answers. Did they identify the same product? Did one hedge more, or add a more useful safety caveat? Neither answer is "grounded" in the real manual — both are reasoning from what the model already knows plus what it sees, so check the specs yourself before trusting either.


In [ ]:
import pandas as pd

claude_result = ask_about_image("claude-sonnet-4-6", CUSTOMER_PHOTO, CUSTOMER_QUESTION)
print(f"--- {claude_result['asked_model']} answered ---")
print(claude_result["answer"])

# Side by side, so you can scan both answers at a glance.
compare = pd.DataFrame([gpt_result, claude_result])
compare["answer_preview"] = compare["answer"].str.replace("\n", " ").str.slice(0, 100) + "…"
compare[["asked_model", "served_model", "answer_preview"]]


## 4 · Turn the photo into a triage record

A free-text answer is nice for the customer, but a support system needs **structured data** to route the ticket automatically — which product, how confident the model is, and what to do next.

> ❓ **Can the same vision call also return clean JSON a ticketing system could consume?**


In [ ]:
import json

TRIAGE_PROMPT = (
    "A customer sent this photo with a product question. Return ONLY a JSON object "
    "with keys: 'product_guess' (your best guess at the product name), "
    "'confidence' ('low'|'medium'|'high'), and 'suggested_next_step' (one short "
    "sentence for a support agent, e.g. confirm the model or send a spec sheet)."
)

triage_raw = ask_about_image("gpt-5.4", CUSTOMER_PHOTO, TRIAGE_PROMPT)["answer"]

# Pull the JSON out of the reply (models sometimes wrap it in prose or code fences).
try:
    triage = json.loads(triage_raw[triage_raw.find("{"): triage_raw.rfind("}") + 1])
    print("✅ Parsed triage record:")
    print(json.dumps(triage, indent=2))
except Exception:
    print("Model didn't return clean JSON — raw reply:")
    print(triage_raw)


## 🧭 Summary — so, can a model turn a photo into a useful answer?

Yes — and you saw three levels of it:

| What you did | What it showed |
|---|---|
| Sent a photo + question to `gpt-5.4` | A vision-capable model can identify a product **and** answer from it in one call |
| Sent the same photo to `claude-sonnet-4-6` | Multimodal input isn't GPT-only — compare style and accuracy across vendors |
| Asked for JSON output | The same vision call can feed a real system (ticket routing), not just chat |

**One caution to carry forward:** none of this was **grounded**. The model reasoned from the image plus its training knowledge — not from Contoso's real product manuals. For that, you need the file-search-backed TrailMate agent from the core labs. Multimodal chat is a great *sandbox* for exploring a capability; a grounded agent is what you'd actually ship.

### Try it yourself
- Swap in a different product photo from `assets/products/` and see if the product guess changes.
- Try a defect-style question ("does this look damaged?") and see how each model hedges.

### Next
➡️ **[02 · Reasoning models](02-reasoning-models.ipynb)** — dial reasoning depth up and down across GPT and Claude on hard Contoso Outdoors planning questions.
